# Configuration

Set the project root directory below (the root of the git repository that was cloned).  Be sure to place this same directory in `DEFAULT:rootdir` in the `bhccpx/BHCCPX.ini` configuration file.

In [1]:
PROJECT_ROOT_DIR = '/home/mf/Projects/mktcpx/coding/bhccpx/'

import os
import sys
os.chdir(os.path.join(PROJECT_ROOT_DIR, 'bhccpx'))
sys.path.append(os.path.join(PROJECT_ROOT_DIR, 'bhccpx'))
os.makedirs('../out/img/', exist_ok=True)

from importlib import reload

#### Core imports

In [4]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

import csv
import zipfile
import pickle as pkl
from pathlib import Path
import argparse

import bhc_datautil
import csv2sys
import sys2bhc
import bhc2out
import bhca
import nic2csv
import logging

#### Load config + common reloads

In [5]:
parser = argparse.ArgumentParser()
bhc_datautil.add_common_args(parser)
args = parser.parse_args([])

config = bhc_datautil.get_config(args, 'notebook')

reload(nic2csv)
reload(csv2sys)
reload(sys2bhc)
reload(bhc2out)
reload(bhca)

<module 'bhca' from '/home/mf/Projects/mktcpx/coding/bhccpx/bhccpx/bhca.py'>

# Reproducing Appendix B
We reproduce the examples in Appendix B of [The Complexity of Bank Holding Companies: A Topological Approach](https://doi.org/10.1016/j.jbankfin.2020.105789). 

### Getting the NIC data
The data for Appendix B in the paper are an older vintage, dated 2016Q2. These older NIC files are available for download in the `data` directory of the **BHCCPX** repo.  Download the five files named `20160630_*.zip`, and extract them someplace reachable locally. 

Once you have done so, place them in the folder you specified in the `DEFAULT:data directory`. You can either unzip them manually (and place the extracted `*.xml` files in the same directory) or run the following code. The resulting `data_files` list contains the names of the extracted files.

### Parsing the NIC data
Given the extracted `*.xml` files, the next step is to parse these into a `*.csv` format.  The following utility:
 * looks for the indicated `*.zip` files in the location configured as `DEFAULT:data directory`, 
 * extracts the `*.xml` files, and 
 * parses those into `*.csv` files. 

In [6]:
# Reload the config, in case it has changed
config = bhc_datautil.get_config(args, 'notebook')

nic2csv.process_files(['20160630*20160701.zip'], config=config)

INFO::nic2csv::36 - Extracted 20160630_TRANSFORMATIONS.XML to /home/mf/Projects/mktcpx/coding/bhccpx/data
INFO::xml2csv::136 - NIC element type for XML file 20160630_TRANSFORMATIONS.XML: TRANSFORMATION


20160630_TRANSFORMATIONS.XML:   0%|          | 0/13329593 [00:00<?, ?it/s]

INFO::nic2csv::36 - Extracted 20160630_ATTRIBUTES_ACTIVE.XML to /home/mf/Projects/mktcpx/coding/bhccpx/data
INFO::xml2csv::136 - NIC element type for XML file 20160630_ATTRIBUTES_ACTIVE.XML: ATTRIBUTES


20160630_ATTRIBUTES_ACTIVE.XML:   0%|          | 0/162038751 [00:00<?, ?it/s]

INFO::nic2csv::36 - Extracted 20160630_RELATIONSHIPS.XML to /home/mf/Projects/mktcpx/coding/bhccpx/data
INFO::xml2csv::136 - NIC element type for XML file 20160630_RELATIONSHIPS.XML: RELATIONSHIP


20160630_RELATIONSHIPS.XML:   0%|          | 0/185813896 [00:00<?, ?it/s]

INFO::nic2csv::36 - Extracted 20160630_ATTRIBUTES_BRANCH.XML to /home/mf/Projects/mktcpx/coding/bhccpx/data
INFO::xml2csv::136 - NIC element type for XML file 20160630_ATTRIBUTES_BRANCH.XML: ATTRIBUTES


20160630_ATTRIBUTES_BRANCH.XML:   0%|          | 0/361688616 [00:00<?, ?it/s]

INFO::nic2csv::36 - Extracted 20160630_ATTRIBUTES_INACTIVE.XML to /home/mf/Projects/mktcpx/coding/bhccpx/data
INFO::xml2csv::136 - NIC element type for XML file 20160630_ATTRIBUTES_INACTIVE.XML: ATTRIBUTES


20160630_ATTRIBUTES_INACTIVE.XML:   0%|          | 0/283202076 [00:00<?, ?it/s]

### Table B.3 Holding company complexities for Wells Fargo & Co. and Wachovia Corp.

### Building the system-wide graphs for configured quarters

In [7]:
# Reload the config, in case it has changed
config = bhc_datautil.get_config(args, 'notebook')

# Converting CSV files to SYS networks
csv2sys.build_sys(config=config)

INFO::csv2sys::244 - Beginning parallel processing (17 tasks across 8 cores) for each as-of date (process messages may be trapped by parallel threads)


Parallel processing per as-of date:   0%|          | 0/17 [00:00<?, ?it/s]

INFO::csv2sys::259 - Parallel processing complete


### Extracting the requested BHCs

Ensure the config file includes at least Wachovia and Wells Fargo:
```
bhclist=[
    1073551, # Wachovia
    1120754, # Wells Fargo
 ]     # NOTE! Leading space required in first column, before bracket
```

And... Ensure the config file includes at least the following four dates:
```
asoflist=[
    '2006Q4',
    '2008Q3',
    '2008Q4',
    '2010Q4',
 ]     # NOTE! Leading space required in first column, before bracket
```

In [8]:
# Reload the config, in case it has changed
config = bhc_datautil.get_config(args, 'notebook')

sys2bhc.process(config=config)

INFO::sys2bhc::333 - Beginning parallel processing (4 tasks across 8 cores) for each as-of date (process messages may be trapped by parallel threads)


Parallel processing for each asofdate:   0%|                                                                   | 0/4 [00:00<?, ?it/s]

Parallel processing for each asofdate: 100%|███████████████████████████████████████████████████████████| 4/4 [00:20<00:00,  5.16s/it]

INFO::sys2bhc::346 - Parallel processing complete


### Create the table

In [11]:
# Reload the config, in case it has changed
config = bhc_datautil.get_config(args, 'notebook')

BHCconfigs = [
    (1073551,20061231),   # Wachovia
    (1073551,20080930),
    (1120754,20061231),   # Wells Fargo
    (1120754,20080930),
    (1120754,20081231),
    (1120754,20101231),
]
df_table = bhc2out.make_bespoke(config)

# Reformat df_table to display the results similarly to JB&F table layout
# Renaming columns to shorter names, including 'Wach' and 'Well' in lieu of RSSD_IDs
column_mapping = {
    '1073551_20061231': 'Wach06Q4',
    '1073551_20080930': 'Wach08Q3',
    '1120754_20061231': 'Well06Q4',
    '1120754_20080930': 'Well08Q3',
    '1120754_20081231': 'Well08Q4',
    '1120754_20101231': 'Well10Q4'
}
# Set the desired column order to match the JB&F table layout
column_order = ['Well06Q4', 'Wach06Q4', 'Well08Q3', 'Wach08Q3', 'Well08Q4', 'Well10Q4']

# Transpose and rename columns in one step
ww_table = df_table.T.rename(columns=column_mapping)[column_order]

# Print table in sections
print("\n")
print(ww_table.iloc[2:5])
print("\n")
print(ww_table.iloc[6:14])
print("\n")
print(ww_table.iloc[14:])

INFO::bhc2out::141 - Creating BHC networks


BHC-Quarter pairs:   0%|          | 0/6 [00:00<?, ?it/s]

INFO::bhc2out::154 - *** Processing bespoke output complete ***


                 Well06Q4 Wach06Q4 Well08Q3 Wach08Q3 Well08Q4 Well10Q4
Bas_Vertex_count      615     1385      620     1037     1720     1552
Bas_Edge_count        654     1674      653     1441     2169     2034
Bas_Cycle_rank         40      290       34      405      450      483


             Well06Q4 Wach06Q4 Well08Q3 Wach08Q3 Well08Q4 Well10Q4
Ent_Qfull_B1      640     1659      639     1427     2153     2019
Ent_Qhetr_B1      225      452      235      672      992      987
Ent_Qfcon_B1       16       17       18       18       27       20
Ent_Qhcon_B1       12       14       14       14       22       16
Ent_edgcn_B1       16        7       12        9       19       15
Ent_DjHom_B1        9       69        8       12       19       10
Ent_DjHom_M       209      247      224      294      578      530
Ent_Nlabl          15       16       15       15       17       16


             Well06Q4 Wach06Q4 Well08Q3 Wac